<a href="https://colab.research.google.com/github/vdlucas-queiroz/CadImobi/blob/main/COTIA_Filtrar_ficticios_por_geometria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import geopandas as gpd

def validar_lotes_complexo(folhas, parametros, lotes, col_folha, col_parametro, col_inscricao):
    """
    Valida consistência de Folha e Parâmetro (subdivisão), aceitando vizinhos adjacentes.

    Flags:
    - flag_folha: Erro na folha/vizinhança
    - flag_parametro: Erro no parâmetro/vizinhança
    - flag_convergencia: 0 se ambos estiverem corretos, 1 se houver qualquer erro.
    """

    # 1. Padronização de CRS
    if lotes.crs != folhas.crs:
        folhas = folhas.to_crs(lotes.crs)
    if lotes.crs != parametros.crs:
        parametros = parametros.to_crs(lotes.crs)

    # 2. Função Interna para criar Dicionário de Adjacências
    def get_adjacencias(gdf, col):
        vizinhos_df = gpd.sjoin(gdf, gdf, predicate='touches', how='left')
        adj = (
            vizinhos_df.groupby(f'{col}_left')[f'{col}_right']
            .apply(lambda x: list(x.dropna().unique()))
            .to_dict()
        )
        # Incluir o próprio código na lista de válidos
        for cod in gdf[col].unique():
            cod_str = str(cod).strip()
            lista = [str(v).strip() for v in adj.get(cod, [])]
            if cod_str not in lista:
                lista.append(cod_str)
            adj[cod] = lista
        return adj

    adj_folhas = get_adjacencias(folhas, col_folha)
    adj_params = get_adjacencias(parametros, col_parametro)

    # 3. Join Espacial (Lotes -> Folhas e Lotes -> Parâmetros)
    # Usamos representative_point para garantir que o lote pertença a apenas 1 polígono de cada
    lotes_pts = lotes.copy()
    lotes_pts['geometry'] = lotes.representative_point()

    # Join com Folhas
    joined_f = gpd.sjoin(lotes_pts[['geometry']], folhas[[col_folha, 'geometry']], how='left', predicate='intersects')
    # Join com Parâmetros
    joined_p = gpd.sjoin(lotes_pts[['geometry']], parametros[[col_parametro, 'geometry']], how='left', predicate='intersects')

    # 4. Lógica de Validação
    def processar_flags(row_index):
        insc = str(lotes.loc[row_index, col_inscricao]).strip()

        # Extração por posição conforme regra: 13461.24...
        # Folha (5 primeiros) | Parâmetro (índice 6 e 7)
        try:
            insc_folha = insc[:5]
            insc_param = insc[6:8]
        except:
            return 1, 1 # Erro de formatação na string

        # Resgate dos códigos espaciais onde o lote caiu
        geo_folha = joined_f.loc[row_index, col_folha]
        geo_param = joined_p.loc[row_index, col_parametro]

        # Validação Folha (inc. vizinhos)
        valid_f = 0 if insc_folha in adj_folhas.get(geo_folha, [str(geo_folha)]) else 1

        # Validação Parâmetro (inc. vizinhos)
        valid_p = 0 if insc_param in adj_params.get(geo_param, [str(geo_param)]) else 1

        return valid_f, valid_p

    # 5. Aplicação das Flags
    flags = [processar_flags(i) for i in lotes.index]

    lotes['flag_folha'] = [f[0] for f in flags]
    lotes['flag_parametro'] = [f[1] for f in flags]

    # Flag consolidada (0 se ambos ok, 1 se algum erro)
    lotes['flag_convergencia'] = (lotes['flag_folha'] | lotes['flag_parametro'])

    return lotes

In [ ]:
# Carregar as folhas
folhas = gpd.read_file("/content/folhas_cit.shp")
lotes = gpd.read_file("/content/lotes_select.shp")
parametro = gpd.read_file("/content/parametro_cit.shp")

# Limpar o caractere "-" da coluna de código da folha
# O parâmetro regex=False torna a operação mais rápida para caracteres simples
folhas['text_min'] = folhas['text_min'].astype(str).str.replace('-', '', regex=False)

In [ ]:
resultado = validar_lotes_complexo(folhas, parametro, lotes, "text_min","quadrante", "cod_geo_l")

In [ ]:
resultado[resultado['flag_convergencia'] == 1]


,gid,cod_geo_l,geom,geometry,flag_folha,flag_parametro,flag_convergencia
29,137505,13432.99.02.0060.00.000,None,"POLYGON ((287908.134 7381013.319, 287904.614 7...",0,1,1
30,137526,13432.99.02.0201.00.000,None,"POLYGON ((287799.953 7380919.795, 287795.687 7...",0,1,1
122,137598,13432.99.02.0243.00.000,None,"POLYGON ((287769.429 7380893.91, 287765.626 73...",0,1,1
181,137617,13432.99.02.0188.00.000,None,"POLYGON ((287797.249 7380949.986, 287813.615 7...",0,1,1
243,387874,23162.33.58.0050.00.000,None,"POLYGON ((306634.183 7388227.231, 306676.874 7...",0,1,1
...,...,...,...,...,...,...,...
64768,198472,23361.14.41.0808.00.000,None,"POLYGON ((303502.781 7376639.098, 303499.321 7...",0,1,1
65890,284288,23253.41.88.0293.00.000,None,"POLYGON ((310644.744 7388638.249, 310643.86 73...",0,1,1
66627,75717,13461.24.07.0001.00.000,None,"POLYGON ((295252.869 7371564.274, 295204.517 7...",1,0,1
67002,67734,13461.24.13.0070.00.000,None,"POLYGON ((294634.674 7371266.671, 294648.623 7...",1,0,1


In [ ]:
ficticios = resultado[resultado['flag_convergencia'] == 1]
ficticios.to_file("ficticios.shp", driver="ESRI Shapefile")
print("Shapefile 'ficticios.shp' exportado com sucesso.")

Shapefile 'ficticios.shp' exportado com sucesso.


/tmp/ipykernel_2412/3307635918.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  ficticios.to_file("ficticios.shp", driver="ESRI Shapefile")
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'flag_parametro' to 'flag_param'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'flag_convergencia' to 'flag_conve'
  ogr_write(
